**Title**: Job Monitoring etc...

**Date**:  TBD

**Description**:  
TBD

### **Requirements**:
1. Access to a Flywheel instance.
3. A Flywheel API key.
2. A Flywheel Project with ideally the dataset used in the [upload-data notebook](https://gitlab.com/flywheel-io/public/flywheel-tutorials/-/blob/master/python/upload-data-to-a-new-project.ipynb).
2. Site Admin Permission?
3. Have some jobs running in your Flywheel Project

<div class="alert alert-block alert-warning" >
    <b>NOTE:</b> This notebook is using a test dataset provided by the <a href="https://gitlab.com/flywheel-io/public/flywheel-tutorials/-/blob/master/python/upload-data-to-a-new-project.ipynb" style="color:black">upload-data notebook</a>. If you have not uploaded this test dataset yet, we strongly recommend you do so now following steps in <a href="https://gitlab.com/flywheel-io/public/flywheel-tutorials/-/blob/master/python/upload-data-to-a-new-project.ipynb" style="color:black">here</a> before proceeding because this notebook is based on a specific project structure.
</div>

<div class="alert alert-block alert-danger" >
    <b>WARNING:</b> The metadata of the acquisitions in your test project will be updated and new files will be created after running the scripts below. 
</div>

# Install and Import Dependencies

In [ ]:
# Install specific packages required for this notebook
!pip install flywheel-sdk pandas

In [1]:
# Import packages
from getpass import getpass
import logging
import os
from pathlib import Path
import re
import datetime
import pprint
from dateutil.tz import tzutc


from IPython.display import display, Image
import flywheel
import pandas as pd
from tqdm import tqdm
from scipy import stats as st
import matplotlib.pyplot as plt
from scipy.stats import normaltest


In [2]:
# Instantiate a logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('root')

# Flywheel API Key and Client

Get a API_KEY. More on this at in the Flywheel SDK doc [here](https://flywheel-io.gitlab.io/product/backend/sdk/branches/master/python/getting_started.html#api-key).

In [3]:
API_KEY = getpass('Enter API_KEY here: ')

Enter API_KEY here:  ····································


Instantiate the Flywheel API client

In [4]:
fw = flywheel.Client(API_KEY or os.environ.get('FW_KEY'))

Show Flywheel logging information

In [5]:
log.info('You are now logged in as %s to %s', fw.get_current_user()['email'], fw.get_config()['site']['api_url'])

2020-06-27 12:45:49,193 INFO You are now logged in as tanxx587@umn.edu to https://ss.ce.flywheel.io:443/api


***

# Initialize a few value

Define your test Project's Label and let's look for it on your Flywheel instance.

In [7]:
PROJECT_LABEL = input('Please enter your Project Label here: ')

Please enter your Project Label here:  anx_test


***

# How to get Job that I launched?

In this section, we will be showing you how to use `get_current_user_jobs` method to get the jobs that you have launched in the past.


Within the Job container, we will be printing the a few attributes within the job such as the `gear_info` that run the job, `state` of the job, and job `id`. 


In [6]:
jobs = fw.get_current_user_jobs()['jobs']


for i, job in enumerate(jobs):
    print(f'Gear Info: {job.gear_info}')
    print(f'Job State: {job.state}')
    print(f'Job ID: {job.id}')
    print()
    if i > 5:
        break
    

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020e1e6f1a

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020a1e6f13

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c401fa1e6f0e

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020a1e6f14

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c4020e1e6f1b

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c401fe1e6f10

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c401e21e70d8



> Expected Output:

```
Gear Info: {'category': 'qa',
 'id': None,
 'name': 'mriqc-demo',
 'version': '0.7.0_0.15.1-hbcd-dev-h'}
Job State: complete
Job ID: 5aee8a5e10a8c402961e70f0

Gear Info: {'category': 'qa',
 'id': None,
 'name': 'mriqc-demo',
 'version': '0.7.0_0.15.1-hbcd-dev-h'}
Job State: complete
Job ID: 5bee77bc10a8c402e21e6f1d
```


<div class="alert alert-block alert-info" style="color: black"><b>INFO:</b> Another method to retrieve jobs is to use <code>get_job()</code>. However, you need to be a Site Admin to run this.</div>

# Find Specific Job with the Job ID

To view a specific job via the Job ID, you can use `get_job_detail`. This will only work for the job you have launched yourself. 

In [ ]:
# Get the latest job that you have launched 
JOB_ID = jobs[0].id

In [44]:
specific_job_detail = fw.get_job_detail(JOB_ID)

In [ ]:
print(f'Gear Info: {specific_job_detail.gear_info}')
print(f'Job State: {specific_job_detail.state}')
print(f'Job ID: {specific_job_detail.id}')

> Expected Output:

```
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc-demo', 'version': '0.7.0_0.15.1-hbcd-dev-h'}
Job State: complete
Job ID: 5bee77bc10a8c402e21e6f1d
```

***

# Filter Job


In this section, we will showcase how to filter job based on the Gear Name, Date and the State of the Job.

## 1. Gear Name



In [23]:
GEAR_NAME = 'mriqc'

In [24]:
def filter_gear(job):
    job_gear = job['gear_info']
    if job_gear['name'] == GEAR_NAME:
        return job
    

In [29]:
filtered_job = list(filter(filter_gear, jobs))

for i, job in enumerate(filtered_job):
    print(f'Gear Info: {job.gear_info}')
    print(f'Job State: {job.state}')
    print(f'Job ID: {job.id}')
    print()
    if i >= 5:
        break

0
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020e1e6f1a

1
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020a1e6f13

2
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c401fa1e6f0e

3
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020a1e6f14

4
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c4020e1e6f1b

5
Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c401fe1e6f10



## 2. Date

In [15]:
CREATED_BY = "2020-06-05"

In [16]:
def filter_date(job):
    if (job.created).strftime("%Y-%m-%d") > CREATED_BY:
        return job
    

In [30]:
filtered_job = list(filter(filter_date, jobs))

for i, job in enumerate(filtered_job):
    print(f'Gear Info: {job.gear_info}')
    print(f'Job State: {job.state}')
    print(f'Job ID: {job.id}')
    print()
    if i >= 5:
        break

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020e1e6f1a

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020a1e6f13

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c401fa1e6f0e

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fa10a8c4020a1e6f14

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c4020e1e6f1b

Gear Info: {'category': 'qa', 'id': None, 'name': 'mriqc', 'version': '0.7.0_0.15.1'}
Job State: failed
Job ID: 5ee555fb10a8c401fe1e6f10



## 3. State of the job

In [8]:
JOB_STATE = 'complete'

In [7]:
def filter_state(job):
    if job.state == JOB_STATE:
        return job
    

In [ ]:
filtered_job = list(filter(filter_state, jobs))

for i, job in enumerate(filtered_job):
    print(f'Gear Info: {job.gear_info}')
    print(f'Job State: {job.state}')
    print(f'Job ID: {job.id}')
    print()
    if i > 5:
        break

***

# Cancelling Job

Simply use the `update` method to cancel the job that is on pending.

In [32]:
JOB_STATE = 'pending'

filtered_job = list(filter(filter_state, jobs))

for job in filtered_job:
    job.update(state='cancelled')
    


***

# Restarting Job

You can also restart a job that has a state of `failed`. However, you can only retry a job once.

In [ ]:
for job in jobs:
    if job.state == 'failed' and job.gear_info['name']:
        fw.retry_job(job.id)
        

***

# Pulling statistics on Jobs:
- states
- average run time
- Getting logs for failed job

## Visualize the Average Run Time for a Specific Gear

In [ ]:
GEAR_NAME = 'mriqc-demo'
CREATED_BY = "2020-06-05"

run_times = list()

for job in jobs:
    if job_gear['name'] == GEAR_NAME and (job.created).strftime("%Y-%m-%d") > CREATED_BY and job.state == "failed":
        job_container = fw.get_job(job.id)
        time_delta = job.transitions.complete - job.transitions.running
        run_times.append(time_delta.total_seconds()/60)
        
if run_times:
    plt.hist(run_times)
    plt.title(f'{GEAR_NAME} run times in minutes')
    plt.show()

## Get the number of jobs based on their state


In [37]:
JOB_STATE = 'pending'

pending_jobs = list(filter(filter_state, jobs))



JOB_STATE = 'running'

running_jobs = list(filter(filter_state, jobs))




In [40]:
print(f'==============================\n{datetime.datetime.now()}\n==============================\n')
print(f'Check Job States \n')
print(f'{len(pending_jobs)} pending jobs \n')
print(f'{len(running_jobs)} running jobs \n')

2020-06-27 13:09:39.154736

Check Job States 

392 pending jobs 

0 running jobs 



In [ ]:
max_run_time = max(run_times) 
min_run_time = min(run_times)
run_time_range = max_run_time - min_run_time
mu = stats.mean(run_times)
sd = stats.stdev(run_times)


# Determine a run_time_cutoff 
s, pval = normaltest(run_times)
if pval < 0.01:
    print('s = {:.2f}. Distribution is normal (enough)... using 2*sd + mu as a cutoff'.format(s))
    run_time_cutoff = 2*sd + mu
else:
    print('s = {:.2f}. Distribution is not normal (enough)... using max time + 1sd as a cutoff'.format(s))
    run_time_cutoff = max_run_time + 1*sd


print('range = {:.2f}\nmu = {:.2f}\nsd = {:.2f}\ncutoff = {:.2f}'.format(run_time_range, mu, sd, run_time_cutoff))
